In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()


DB_USER = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")
DB_HOST = os.getenv("DB_HOST", "127.0.0.1")
DB_PORT = os.getenv("DB_PORT", "3306")
DB_NAME = os.getenv("DB_NAME", "testdb")

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, engine)

In [2]:
# Build motherfiles of all cycle time definitions
window_start = "2018-01-01 00:00:00"
window_end = "2020-10-31 23:59:59"
project_keys = ["SERVER", "MDL", "JRACLOUD"]

mother_dir = Path("../reports/mother_files")
mother_dir.mkdir(parents=True, exist_ok=True)

summary_rows = []

for project_key in project_keys:
    sql = f"""
    WITH base_issues AS (
      SELECT
        p.Project_Key AS Project_Key,
        i.ID AS Issue_ID,
        i.Jira_ID,
        i.Issue_Key,
        i.URL AS Issue_URL,
        i.Title,
        i.Type AS Issue_Type,
        i.Priority,
        i.Status AS Current_Status,
        i.Resolution AS Current_Resolution,
        i.Creation_Date AS Created,
        i.Estimation_Date,
        i.Resolution_Date AS Resolution,
        i.Last_Updated,
        i.Story_Point,
        i.Timespent,
        i.Total_Effort_Minutes,
        i.Resolution_Time_Minutes,
        i.In_Progress_Minutes,
        i.Title_Changed_After_Estimation,
        i.Description_Changed_After_Estimation,
        i.Story_Point_Changed_After_Estimation,
        i.Pull_Request_URL,
        i.Sprint_ID,
        s.Name AS Sprint_Name,
        s.State AS Sprint_State,
        s.Start_Date AS Sprint_Start_Date,
        s.End_Date AS Sprint_End_Date,
        s.Activated_Date AS Sprint_Activated_Date,
        s.Complete_Date AS Sprint_Complete_Date
      FROM Issue i
      JOIN Project p
        ON p.ID = i.Project_ID
      LEFT JOIN Sprint s
        ON s.ID = i.Sprint_ID
      WHERE p.Project_Key = '{project_key}'
        AND i.Creation_Date BETWEEN '{window_start}' AND '{window_end}'
        AND i.Resolution_Date IS NOT NULL
    ),
    status_events AS (
      SELECT
        bi.Project_Key,
        bi.Issue_ID,
        cl.ID AS Change_Log_ID,
        cl.Creation_Date AS Change_Date,

        COALESCE(NULLIF(cl.From_String, ''), cl.From_Value) AS From_Status_Raw,
        COALESCE(NULLIF(cl.To_String, ''), cl.To_Value) AS To_Status_Raw,

        LOWER(TRIM(COALESCE(NULLIF(cl.From_String, ''), cl.From_Value))) AS From_Status,
        LOWER(TRIM(COALESCE(NULLIF(cl.To_String, ''), cl.To_Value))) AS To_Status
      FROM base_issues bi
      JOIN Change_Log cl
        ON cl.Issue_ID = bi.Issue_ID
      WHERE LOWER(TRIM(cl.Field)) = 'status'
    ),
    first_status_event AS (
      SELECT
        x.Project_Key,
        x.Issue_ID,
        x.Change_Date AS First_Status_Transition,
        x.To_Status_Raw AS First_Status_To_Raw,
        x.To_Status AS First_Status_To
      FROM (
        SELECT
          se.*,
          ROW_NUMBER() OVER (
            PARTITION BY se.Project_Key, se.Issue_ID
            ORDER BY se.Change_Date, se.Change_Log_ID
          ) AS rn
        FROM status_events se
      ) x
      WHERE x.rn = 1
    ),
    status_rollup AS (
      SELECT
        se.Project_Key,
        se.Issue_ID,
        COUNT(*) AS Num_Status_Changes,
        MIN(CASE
              WHEN se.To_Status = 'in progress'
                OR se.To_Status = 'inprogress'
                OR se.To_Status LIKE '%%in progress%%'
              THEN se.Change_Date
            END) AS First_In_Progress_Timestamp,
        MIN(CASE
              WHEN se.To_Status IN ('done', 'closed', 'resolved', 'verified')
              THEN se.Change_Date
            END) AS First_Done_Timestamp,
        MAX(CASE
              WHEN se.To_Status IN ('done', 'closed', 'resolved', 'verified')
              THEN se.Change_Date
            END) AS Final_Done_Timestamp,
        MAX(CASE
              WHEN se.From_Status IN ('done', 'closed', 'resolved', 'verified')
               AND se.To_Status NOT IN ('done', 'closed', 'resolved', 'verified')
              THEN 1 ELSE 0
            END) AS Was_Reopened
      FROM status_events se
      GROUP BY se.Project_Key, se.Issue_ID
    )
    SELECT
      bi.Project_Key,
      bi.Issue_ID,
      bi.Jira_ID,
      bi.Issue_Key,
      bi.Issue_URL,
      bi.Title,
      bi.Issue_Type,
      bi.Priority,
      bi.Current_Status,
      bi.Current_Resolution,

      bi.Created,
      bi.Estimation_Date,
      bi.Resolution,
      bi.Last_Updated,

      bi.Story_Point,
      bi.Timespent,
      bi.Total_Effort_Minutes,
      bi.Resolution_Time_Minutes,
      bi.In_Progress_Minutes,

      bi.Title_Changed_After_Estimation,
      bi.Description_Changed_After_Estimation,
      bi.Story_Point_Changed_After_Estimation,
      bi.Pull_Request_URL,

      bi.Sprint_ID,
      bi.Sprint_Name,
      bi.Sprint_State,
      bi.Sprint_Start_Date,
      bi.Sprint_End_Date,
      bi.Sprint_Activated_Date,
      bi.Sprint_Complete_Date,

      (sr.Issue_ID IS NOT NULL) AS Has_Status_History,
      COALESCE(sr.Num_Status_Changes, 0) AS Num_Status_Changes,

      fse.First_Status_Transition,
      fse.First_Status_To_Raw,
      fse.First_Status_To,

      sr.First_In_Progress_Timestamp,
      sr.First_Done_Timestamp,
      sr.Final_Done_Timestamp,
      COALESCE(sr.Was_Reopened, 0) AS Was_Reopened,

      bi.Resolution_Time_Minutes AS CT1_Resolution_Time_Minutes,
      bi.In_Progress_Minutes AS CT2_In_Progress_Minutes,
      TIMESTAMPDIFF(MINUTE, bi.Created, bi.Resolution) AS CT3_Resolution_minus_Created_Minutes,
      TIMESTAMPDIFF(MINUTE, sr.First_In_Progress_Timestamp, bi.Resolution) AS CT4_Resolution_minus_FirstInProgress_Minutes,
      TIMESTAMPDIFF(MINUTE, sr.First_In_Progress_Timestamp, sr.Final_Done_Timestamp) AS CT5_FinalDone_minus_FirstInProgress_Minutes,
      TIMESTAMPDIFF(MINUTE, sr.First_In_Progress_Timestamp, sr.First_Done_Timestamp) AS CT6_FirstDone_minus_FirstInProgress_Minutes,

      (bi.Resolution IS NOT NULL AND bi.Resolution_Time_Minutes IS NOT NULL) AS Has_CT1,
      (bi.Resolution IS NOT NULL AND bi.In_Progress_Minutes IS NOT NULL) AS Has_CT2,
      (bi.Created IS NOT NULL AND bi.Resolution IS NOT NULL) AS Has_CT3,
      (sr.First_In_Progress_Timestamp IS NOT NULL AND bi.Resolution IS NOT NULL) AS Has_CT4,
      (sr.First_In_Progress_Timestamp IS NOT NULL AND sr.Final_Done_Timestamp IS NOT NULL AND bi.Resolution IS NOT NULL) AS Has_CT5,
      (sr.First_In_Progress_Timestamp IS NOT NULL AND sr.First_Done_Timestamp IS NOT NULL AND bi.Resolution IS NOT NULL) AS Has_CT6

    FROM base_issues bi
    LEFT JOIN status_rollup sr
      ON sr.Project_Key = bi.Project_Key
     AND sr.Issue_ID = bi.Issue_ID
    LEFT JOIN first_status_event fse
      ON fse.Project_Key = bi.Project_Key
     AND fse.Issue_ID = bi.Issue_ID
    ORDER BY bi.Created, bi.Issue_ID;
    """

    df_mother = q(sql)

    out = mother_dir / f"mother_{project_key}_2018_2020.csv"
    df_mother.to_csv(out, index=False)

    summary_rows.append({
        "Project": project_key,
        "Issues": len(df_mother),
        "Has_Status_History": int(df_mother["Has_Status_History"].fillna(0).astype(int).sum()),
        "Has_CT1": int(df_mother["Has_CT1"].fillna(0).astype(int).sum()),
        "Has_CT2": int(df_mother["Has_CT2"].fillna(0).astype(int).sum()),
        "Has_CT3": int(df_mother["Has_CT3"].fillna(0).astype(int).sum()),
        "Has_CT4": int(df_mother["Has_CT4"].fillna(0).astype(int).sum()),
        "Has_CT5": int(df_mother["Has_CT5"].fillna(0).astype(int).sum()),
        "Has_CT6": int(df_mother["Has_CT6"].fillna(0).astype(int).sum()),
        "Output_File": str(out),
    })

mother_summary = pd.DataFrame(summary_rows)
mother_summary

,Project,Issues,Has_Status_History,Has_CT1,Has_CT2,Has_CT3,Has_CT4,Has_CT5,Has_CT6,Output_File
0,SERVER,15974,15974,15974,15974,15974,9215,9215,9215,../reports/mother_files/mother_SERVER_2018_202...
1,MDL,4901,4901,4901,4901,4901,3489,3489,3489,../reports/mother_files/mother_MDL_2018_2020.csv
2,JRACLOUD,1646,1646,1646,1646,1646,275,275,275,../reports/mother_files/mother_JRACLOUD_2018_2...
